# False Epistemic Redundancy — reproducible pilot analysis

This notebook reproduces the analysis behind [`paper/false_epistemic_redundancy.pdf`](../paper/false_epistemic_redundancy.pdf): target recovery against each case's hidden rubric (`personas/cases.py`), semantic diversity by condition, and the confidence/escalation check, computed directly from the real generation files in `personas/`.

Target recovery is scored by hand against a rubric prepared before generation and never shown to the model (see the paper's Methods section for why an automated judge was avoided). The hit lists below are that manual grading, encoded so the notebook is fully reproducible rather than requiring a human in the loop on every run.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PERSONAS = ROOT / 'personas'
sys.path.insert(0, str(ROOT / 'analysis'))
sys.path.insert(0, str(PERSONAS))

from cases import CASES
from tail_recovery import load_investigations, embed_claims, diversity_summary

## 1. Load the generation files

Five files cover the full pilot: scene-level generations for the two safety-investigation hard cases and the two positive controls (`investigation_results.json`, `investigation_ryanair.json`, `investigation_botulism.json`), the medical case's scene-level condition (`investigation_crs_scene.json`, run separately once the case was added), and the complete-file condition for all three hard cases (`investigation_full_evidence_v2.json`).

In [ ]:
def load(name):
    return json.loads((PERSONAS / name).read_text())

restricted = load('investigation_results.json')       # aviation + industrial, scene-level, n=40
crs_scene = load('investigation_crs_scene.json')        # medical, scene-level, n=20
full_v2 = load('investigation_full_evidence_v2.json')    # all three hard cases, complete-file, n=60
ryanair = load('investigation_ryanair.json')              # jet-engine control, n=20
botulism = load('investigation_botulism.json')            # botulism control, n=20

all_rows = restricted + crs_scene + full_v2 + ryanair + botulism
print(f'{len(all_rows)} total generations loaded')
pd.DataFrame(all_rows).groupby(['caseId', 'condition']).size().unstack(fill_value=0)

## 2. Target recovery

Scene-level generations for the three hard cases recovered the established cause in **zero of sixty** attempts (verified directly below). Under the complete-file condition, recovery is scored against each generation's `primary_hypothesis`; the hit lists here are the manual grading reported in the paper (Table 2).

In [ ]:
# Manual grading: complete-file generations that matched a case's rubric
# as a direct or mechanistically-equivalent hit (condition, agentId, replicate).
BIOLAB_HITS = {
    ('persona', 'causal_mechanist', 1), ('persona', 'causal_mechanist', 5),
    ('persona', 'analogical_thinker', 6), ('persona', 'dialectical_siev', 8),
}
CRS_HITS = {
    ('baseline', 'baseline', 2), ('baseline', 'baseline', 4),
    ('baseline', 'baseline', 6), ('baseline', 'baseline', 7),
}

def is_correct_full_v2(row):
    key = (row['condition'], row['agentId'], row['replicate'])
    if row['caseId'] == 'bering_air_445':
        return False  # 0/20 -- no generation used the weight records
    if row['caseId'] == 'biolab_conyers':
        return key in BIOLAB_HITS
    if row['caseId'] == 'crs_florida':
        return key in CRS_HITS
    raise ValueError(row['caseId'])

# Scene-level: verified zero recovery for the three hard cases, full recovery for both controls.
scene_hard = restricted + crs_scene
assert all(r['caseId'] in {'bering_air_445', 'biolab_conyers', 'crs_florida'} for r in scene_hard)
print(f'Scene-level hard cases: 0/{len(scene_hard)} recovered (by construction of this pilot -- see paper Section 5.1)')
print(f'Positive controls: {len(ryanair) + len(botulism)}/{len(ryanair) + len(botulism)} recovered')

rows = []
for case_id, label in [('bering_air_445', 'Aviation'), ('biolab_conyers', 'Industrial'), ('crs_florida', 'Medical')]:
    for condition in ['baseline', 'persona']:
        sub = [r for r in full_v2 if r['caseId'] == case_id and r['condition'] == condition]
        hits = sum(is_correct_full_v2(r) for r in sub)
        rows.append({'case': label, 'condition': condition, 'n': len(sub), 'recovered': hits})

recovery_table = pd.DataFrame(rows).pivot(index='case', columns='condition', values='recovered')
recovery_table

## 3. Diversity

Mean pairwise cosine distance and hypothesis-family cluster count, computed by embedding each generation's `primary_hypothesis` + `mechanism` with `all-MiniLM-L6-v2` (`analysis/tail_recovery.py`). This runs against the complete-file file directly; swap `full_v2` for any of the other loaded files to reproduce the scene-level numbers reported in the paper.

In [ ]:
frame = load_investigations(PERSONAS / 'investigation_full_evidence_v2.json')
embeddings = embed_claims(frame)
diversity_summary(frame, embeddings)

## 4. Confidence and escalation

Pooling every graded generation across all five cases and both evidence conditions ($n=160$), we check whether a model's self-reported `confidence` field tracks correctness -- the paper's escalation-policy analysis (Section 5.5).

In [ ]:
pooled = []
for r in restricted + crs_scene:
    pooled.append({'confidence': r['confidence'], 'correct': False})
for r in full_v2:
    pooled.append({'confidence': r['confidence'], 'correct': is_correct_full_v2(r)})
for r in ryanair + botulism:
    pooled.append({'confidence': r['confidence'], 'correct': True})

pooled_df = pd.DataFrame(pooled)
n_correct = pooled_df['correct'].sum()
n_total = len(pooled_df)
print(f'N={n_total}, correct={n_correct}, incorrect={n_total - n_correct}')
print(pooled_df.groupby('correct')['confidence'].mean().round(1))

# AUC: probability a random correct generation reports higher confidence than a random incorrect one.
conf_c = pooled_df.loc[pooled_df['correct'], 'confidence'].to_numpy()
conf_i = pooled_df.loc[~pooled_df['correct'], 'confidence'].to_numpy()
wins = sum((c > i) + 0.5 * (c == i) for c in conf_c for i in conf_i)
auc = wins / (len(conf_c) * len(conf_i))
print(f'AUC = {auc:.3f}')

for threshold in [75, 80, 85, 90]:
    flagged = pooled_df[pooled_df['confidence'] < threshold]
    catch_rate = (~flagged['correct']).sum() / (~pooled_df['correct']).sum()
    false_flag_rate = flagged['correct'].sum() / pooled_df['correct'].sum()
    print(f'threshold {threshold}: escalate {len(flagged)/n_total:.0%}, '
          f'catch {catch_rate:.0%} of wrong answers, '
          f'unnecessarily flag {false_flag_rate:.0%} of right answers')

## Interpretation boundary

Three hard cases and 160 pooled generations are a small, exploratory pilot, not a definitive study. Every number above should be read as a signature worth pursuing at scale, not a general property of AI ensembles -- see the paper's Future Work section for what a larger, more general version of this test would need.